# ViNLI (COLING 2022) — Kaggle GPU reproduction

Notebook tái lập gần nhất các baseline Transformer trong *ViNLI: A Vietnamese Corpus for Studies on Open-Domain Natural Language Inference* (Huynh et al., COLING 2022). Mặc định chạy **XLM-R Large, 4 nhãn** — cấu hình có điểm số Test cao nhất trong paper.

Paper: https://aclanthology.org/2022.coling-1.339/  \nResearch/protocol: `docs/research/2026-09-19-vinli-coling2022-reproduction.md`

> Corpus gốc và code/checkpoint của tác giả hiện không còn được công khai từ nguồn chính thức. Hãy tự đính kèm corpus ViNLI mà bạn có quyền dùng vào Kaggle Dataset; không đưa corpus vào Git.

## 1. Bật GPU và gắn dữ liệu

Trong Kaggle: **Settings → Accelerator → GPU**, rồi đính kèm Kaggle Dataset chứa `train`, `dev`, `test` (đuôi `.json`, `.jsonl`, hoặc `.csv`). Mỗi bản ghi phải có premise/hypothesis/label; notebook chấp nhận thêm các tên `sentence1`, `sentence2`, `gold_label`, `relation`. Nhãn chấp nhận `E/C/N/O` hoặc `entailment/contradiction/neutral/other`.

Nếu bật Internet để tải checkpoint lần đầu, sau đó có thể lưu notebook output/cache để chạy offline.

In [ ]:
!pip -q install -U 'transformers>=4.38,<5' 'accelerate>=0.27' 'scikit-learn>=1.2,<1.9' huggingface_hub

import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

assert torch.cuda.is_available(), 'Hãy bật Kaggle GPU trước khi train.'
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)

## 1.1. Tải corpus mirror để chạy replication

Mirror `presencesw/vinli_4_label` trên Hugging Face có đúng split count Table 9 và các cột cần thiết. Đây là **third-party mirror**, không phải bản tác giả phát hành; notebook lưu provenance để kết quả được gọi đúng là replication, không phải exact reproduction.

In [ ]:
from huggingface_hub import hf_hub_download

MIRROR_REPO = 'presencesw/vinli_4_label'
DATA_DIR = Path('/kaggle/working/vinli_4_label')
DATA_DIR.mkdir(parents=True, exist_ok=True)

for split in ('train', 'dev', 'test'):
    target = DATA_DIR / f'{split}.json'
    if not target.exists():
        parquet_path = hf_hub_download(
            repo_id=MIRROR_REPO, repo_type='dataset',
            filename=f'data/{split}-00000-of-00001.parquet',
        )
        pd.read_parquet(parquet_path).to_json(
            target, orient='records', force_ascii=False
        )

provenance = {
    'source_type': 'third_party_mirror',
    'source': f'https://huggingface.co/datasets/{MIRROR_REPO}',
    'expected_paper_counts': {'train': 24376, 'dev': 3009, 'test': 2991},
    'warning': 'Not an author-owned or officially verified ViNLI release.',
}
(DATA_DIR / 'provenance.json').write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
print('Downloaded mirror to:', DATA_DIR)
print({split: len(pd.read_json(DATA_DIR / f'{split}.json')) for split in ('train', 'dev', 'test')})

In [ ]:
# DATA_DIR is created by the preceding mirror-download cell.
OUTPUT_DIR = Path('/kaggle/working/vinli-xlmr-large-4label')

# Paper: Adam, lr=1e-5, batch_size=16, epochs=10 cho Transformer.
MODEL_KEY = 'xlmr-large'  # mbert | xlmr-base | xlmr-large | phobert-base | phobert-large
NUM_LABELS = 4            # đổi thành 3 để lặp lại setting không có OTHER
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-5
SEED = 42                 # paper không công bố seed
# Paper không công bố max sequence length cho Transformer. 128 bao phủ rất rộng
# so với trung bình premise/hypothesis 24.5/18.1 words; tăng lên 256 nếu corpus cho thấy truncation.
MAX_LENGTH = 128

MODEL_IDS = {
    'mbert': 'bert-base-multilingual-cased',
    'xlmr-base': 'xlm-roberta-base',
    'xlmr-large': 'xlm-roberta-large',
    'phobert-base': 'vinai/phobert-base',
    'phobert-large': 'vinai/phobert-large',
}
assert MODEL_KEY in MODEL_IDS
assert NUM_LABELS in (3, 4)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_4 = ('entailment', 'contradiction', 'neutral', 'other')
LABELS = LABELS_4[:NUM_LABELS]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
print({'model': MODEL_IDS[MODEL_KEY], 'labels': LABELS, 'output': str(OUTPUT_DIR)})

In [ ]:
ALIASES = {
    'premise': ('premise', 'sentence1', 'sentence_1', 'text_a'),
    'hypothesis': ('hypothesis', 'sentence2', 'sentence_2', 'text_b'),
    'label': ('label', 'gold_label', 'relation', 'inference_label'),
    'topic': ('topic', 'domain', 'category'),
}
NORMALIZE_LABEL = {
    'e': 'entailment', 'entailment': 'entailment', 'entails': 'entailment',
    'c': 'contradiction', 'contradiction': 'contradiction', 'contradicts': 'contradiction',
    'n': 'neutral', 'neutral': 'neutral',
    'o': 'other', 'other': 'other',
}

def find_split(name):
    matches = []
    for suffix in ('json', 'jsonl', 'csv'):
        matches.extend(DATA_DIR.rglob(f'{name}.{suffix}'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {name}.json/.jsonl/.csv beneath {DATA_DIR}; found {matches}')
    return matches[0]

def read_records(path):
    if path.suffix == '.jsonl':
        return [json.loads(line) for line in path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
    if path.suffix == '.csv':
        return pd.read_csv(path).to_dict('records')
    payload = json.loads(path.read_text(encoding='utf-8-sig'))
    if isinstance(payload, dict):
        payload = next((payload[key] for key in ('data', 'examples', 'records') if key in payload), payload)
    if not isinstance(payload, list):
        raise ValueError(f'{path} must be a list or contain data/examples/records')
    return payload

def value(record, field):
    for key in ALIASES[field]:
        if key in record and pd.notna(record[key]):
            return record[key]
    if field == 'topic':
        return None
    raise ValueError(f'Missing {field}; expected one of {ALIASES[field]}')

def load_split(name):
    path = find_split(name)
    rows = []
    for row_number, record in enumerate(read_records(path), start=1):
        try:
            label = NORMALIZE_LABEL[str(value(record, 'label')).strip().lower()]
            if label not in LABEL_TO_ID:  # three-label experiment removes OTHER rather than relabeling it
                continue
            premise, hypothesis = str(value(record, 'premise')).strip(), str(value(record, 'hypothesis')).strip()
            if not premise or not hypothesis:
                raise ValueError('empty premise/hypothesis')
            rows.append({'premise': premise, 'hypothesis': hypothesis, 'label': label, 'topic': value(record, 'topic')})
        except Exception as exc:
            raise ValueError(f'{path}, record {row_number}: {exc}') from exc
    return pd.DataFrame(rows), path

splits, paths = {}, {}
for split in ('train', 'dev', 'test'):
    splits[split], paths[split] = load_split(split)
    print(split, paths[split], len(splits[split]), Counter(splits[split].label))

# Paper's four-label reference totals: 24,376 / 3,009 / 2,991.
# A mismatch is not automatically an error: verify corpus release/split before comparing scores.
pairs = {name: set(zip(frame.premise.str.lower().str.split().str.join(' '), frame.hypothesis.str.lower().str.split().str.join(' '))) for name, frame in splits.items()}
for left, right in (('train', 'dev'), ('train', 'test'), ('dev', 'test')):
    print(f'exact normalized overlap {left}/{right}:', len(pairs[left] & pairs[right]))

In [ ]:
# XLM-R/mBERT dùng fast tokenizer để mã hóa batch nhanh; PhoBERT không có fast tokenizer tương đương.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_IDS[MODEL_KEY],
    use_fast=not MODEL_KEY.startswith('phobert'),
)

class VinliPairs(Dataset):
    def __init__(self, frame):
        self.labels = torch.tensor([LABEL_TO_ID[x] for x in frame.label], dtype=torch.long)
        self.encodings = tokenizer(
            frame.premise.tolist(), frame.hypothesis.tolist(),
            truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt'
        )
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {name: tensor[index] for name, tensor in self.encodings.items()} | {'labels': self.labels[index]}

datasets = {name: VinliPairs(frame) for name, frame in splits.items()}
loaders = {
    'train': DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True, pin_memory=True),
    'dev': DataLoader(datasets['dev'], batch_size=BATCH_SIZE, pin_memory=True),
    'test': DataLoader(datasets['test'], batch_size=BATCH_SIZE, pin_memory=True),
}
print({name: len(dataset) for name, dataset in datasets.items()})

In [ ]:
device = torch.device('cuda')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_IDS[MODEL_KEY], num_labels=NUM_LABELS
).to(device)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)  # paper reports Adam

@torch.no_grad()
def evaluate(loader):
    model.eval()
    gold, prediction = [], []
    for batch in loader:
        labels = batch.pop('labels').to(device)
        logits = model(**{name: tensor.to(device) for name, tensor in batch.items()}).logits
        gold.extend(labels.cpu().tolist())
        prediction.extend(logits.argmax(-1).cpu().tolist())
    return {
        'count': len(gold),
        'accuracy': accuracy_score(gold, prediction),
        'macro_f1': f1_score(gold, prediction, average='macro'),
        'per_label': classification_report(gold, prediction, labels=list(range(NUM_LABELS)), target_names=LABELS, output_dict=True, zero_division=0),
    }

history, best_dev_accuracy = [], -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for batch in loaders['train']:
        labels = batch.pop('labels').to(device)
        output = model(**{name: tensor.to(device) for name, tensor in batch.items()}, labels=labels)
        output.loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        losses.append(output.loss.item())
    dev_result = evaluate(loaders['dev'])
    row = {'epoch': epoch, 'train_loss': float(np.mean(losses)), 'dev': dev_result}
    history.append(row)
    print(f"epoch={epoch:02d} loss={row['train_loss']:.4f} dev_acc={dev_result['accuracy']:.4f} dev_f1={dev_result['macro_f1']:.4f}")
    if dev_result['accuracy'] > best_dev_accuracy:
        best_dev_accuracy = dev_result['accuracy']
        model.save_pretrained(OUTPUT_DIR / 'checkpoint-best')
        tokenizer.save_pretrained(OUTPUT_DIR / 'checkpoint-best')

In [ ]:
# Report test only from the checkpoint selected on Dev.
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR / 'checkpoint-best').to(device)
best_dev = max(history, key=lambda row: row['dev']['accuracy'])['dev']
test_result = evaluate(loaders['test'])
result = {
    'paper': 'Huynh et al. (2022), ViNLI',
    'model_key': MODEL_KEY, 'model_id': MODEL_IDS[MODEL_KEY],
    'label_set': NUM_LABELS, 'labels': LABELS,
    'hyperparameters': {'optimizer': 'Adam', 'learning_rate': LEARNING_RATE, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'max_length': MAX_LENGTH, 'seed': SEED},
    'data_paths': {key: str(value) for key, value in paths.items()},
    'split_sizes': {key: len(value) for key, value in splits.items()},
    'history': history, 'best_dev': best_dev, 'test': test_result,
}
(OUTPUT_DIR / 'run.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame([{'split': 'dev', **{key: best_dev[key] for key in ('accuracy', 'macro_f1')}}, {'split': 'test', **{key: test_result[key] for key in ('accuracy', 'macro_f1')}}])

## 2. Đối chiếu kết quả

Mốc Table 6 của paper cho **XLM-R Large**: 3 nhãn Test Acc/F1 = **81.36/81.31**, 4 nhãn Test Acc/F1 = **85.99/86.10**. Các số này chỉ so sánh được nếu corpus, split, tiền xử lý, base checkpoint, seed và lựa chọn checkpoint tương đương. Paper không công bố seed, cách chọn checkpoint, max length Transformer, bản phát hành corpus/split hoặc code huấn luyện; do đó notebook ghi rõ các giá trị thực tế trong `run.json` thay vì khẳng định tái lập tuyệt đối.

PhoBERT trong paper dùng word-level input (VnCoreNLP); trước khi chạy `phobert-*`, cần word-segment premise/hypothesis nhất quán và xác nhận format corpus. XLM-R/mBERT dùng raw Vietnamese syllable-spaced text.